# Velocity Scale Ablation Study

This notebook compares the velocity-scale ablation runs from Weights & Biases.

Each plot shows one metric against **epoch** up to epoch **1100**. The compared runs are:

- No scaling
- 1/(4π)
- 1/(2π)
- 1/π
- 2/π

The plots use high-contrast colors and different line styles for easier comparison.

In [ ]:
from __future__ import annotations

import pandas as pd
import matplotlib.pyplot as plt
import wandb

MAX_EPOCH = 1100
ENTITY = "xanderbaatz-danmarks-tekniske-universitet-dtu"

RUNS = [
    {
        "label": "1/(2π)",
        "project": "mp20_lattice_eps_em",
        "run_id": "w41cll5b",
        "color": "#000000",
        "linestyle": "-",
    },
    {
        "label": "2/π",
        "project": "AblationStudyVelScale_2OverPi",
        "run_id": "pmd5hauj",
        "color": "#0072B2",
        "linestyle": "--",
    },
    {
        "label": "1/π",
        "project": "AblationStudyVelScale_1Pi",
        "run_id": "k57g5zrv",
        "color": "#D55E00",
        "linestyle": "-.",
    },
    {
        "label": "1/(4π)",
        "project": "AblationStudyVelScale_4Pi",
        "run_id": "3usx62u3",
        "color": "#009E73",
        "linestyle": ":",
    },
    {
        "label": "No scaling",
        "project": "AblationStudyVelScale_NoScale",
        "run_id": "dgbi4imd",
        "color": "#CC79A7",
        "linestyle": (0, (5, 2, 1, 2)),
    },
]

METRICS = [
    "train/loss_v",
    "train/loss_l",
    "train/loss_weighted",
    "val/rmse",
    "val/valid",
    "val/match_rate",
]

api = wandb.Api()

In [ ]:
def fetch_run_data(project: str, run_id: str) -> pd.DataFrame:
    run = api.run(f"{ENTITY}/{project}/{run_id}")
    history = run.history(keys=["epoch", *METRICS], pandas=True)
    history = history.dropna(subset=["epoch"]).copy()
    history = history[history["epoch"] <= MAX_EPOCH]
    history["epoch"] = history["epoch"].astype(float)
    return history


all_histories = {
    f"{run['project']}/{run['run_id']}": fetch_run_data(run["project"], run["run_id"])
    for run in RUNS
}


def plot_metric(metric: str, title: str, annotate_last: bool = False) -> None:
    plt.figure(figsize=(10, 6))

    endpoints: list[tuple[float, float, str, dict[str, object]]] = []

    for run in RUNS:
        run_key = f"{run['project']}/{run['run_id']}"
        history = all_histories.get(run_key)
        if history is None or history.empty or metric not in history.columns:
            continue

        subset = history[["epoch", metric]].dropna().sort_values("epoch")
        if subset.empty:
            continue

        style = {
            "color": run["color"],
            "linestyle": run["linestyle"],
        }
        plt.plot(
            subset["epoch"],
            subset[metric],
            label=run["label"],
            linewidth=2.2,
            **style,
        )

        if annotate_last:
            last = subset.iloc[-1]
            endpoints.append((
                float(last["epoch"]),
                float(last[metric]),
                run["label"],
                style,
            ))

    if annotate_last and endpoints:
        endpoints.sort(key=lambda item: item[1])
        y_positions = [item[1] for item in endpoints]
        y_min = min(y_positions)
        y_max = max(y_positions)
        span = max(y_max - y_min, 1e-6)
        min_gap = max(span * 0.06, 0.01)

        adjusted = []
        for value in y_positions:
            if not adjusted:
                adjusted.append(value)
            else:
                adjusted.append(max(value, adjusted[-1] + min_gap))

        if adjusted:
            upper = y_max + min_gap
            overflow = adjusted[-1] - upper
            if overflow > 0:
                adjusted = [value - overflow for value in adjusted]
                lower = y_min - min_gap
                underflow = lower - adjusted[0]
                if underflow > 0:
                    adjusted = [value + underflow for value in adjusted]

        x_max = max(item[0] for item in endpoints)
        x_text = x_max + max(5.0, x_max * 0.02)

        for (x, original_y, label, style), y_text in zip(endpoints, adjusted):
            color = style.get("color", "black")
            plt.annotate(
                f"{label}: {original_y:.4f}",
                xy=(x, original_y),
                xytext=(x_text, y_text),
                textcoords="data",
                color=color,
                fontsize=10,
                va="center",
                arrowprops={"arrowstyle": "-", "color": color, "lw": 1.0},
                bbox={"boxstyle": "round,pad=0.2", "fc": "white", "ec": color, "alpha": 0.9},
                clip_on=False,
            )

        plt.xlim(right=x_text + max(10.0, x_max * 0.05))

    plt.title(title)
    plt.xlabel("Epoch")
    plt.ylabel(metric)
    plt.grid(True, alpha=0.3)
    plt.legend(frameon=True)
    plt.tight_layout()
    plt.show()


In [ ]:
summary_rows = []
for run in RUNS:
    label = run["label"]
    run_key = f"{run['project']}/{run['run_id']}"
    history = all_histories.get(run_key)
    row = {"run": label}

    for metric in METRICS:
        if history is None or history.empty or metric not in history.columns:
            row[metric] = None
            continue

        series = history[[metric]].dropna()
        row[metric] = None if series.empty else float(series.iloc[-1][metric])

    summary_rows.append(row)

pd.DataFrame(summary_rows)


In [ ]:
plot_metric('train/loss_v', 'Velocity Training Loss')

In [ ]:
plot_metric('train/loss_l', 'Lattice Training Loss')

In [ ]:
plot_metric('train/loss_weighted', 'Weighted Training Loss')

In [ ]:
plot_metric('val/rmse', 'Validation RMSE', annotate_last=True)

In [ ]:
plot_metric('val/valid', 'Validation Valid Rate', annotate_last=True)

In [ ]:
plot_metric('val/match_rate', 'Validation Match Rate', annotate_last=True)